# ⚡ MoneyPrinter Turbo - Google Colab & Google Drive Video Üretim Stüdyosu

> **NVIDIA GPU Donanım Hızlandırmalı (NVENC), Google Drive Entegre ve Kesintisiz Auto-Resume Destekli Video Üretim Sistemi**

Bu Colab Notebook ile MoneyPrinter Turbo Lite sürümünü çalıştırabilir; **NVIDIA GPU donanım hızlandırma ile saniyeler içinde 4K/1080p video üretebilir** ve tüm ayarlarınızı, görevlerinizi ve biten videolarınızı doğrudan **Google Drive**'ınızda saklayabilirsiniz.

### 🌟 Öne Çıkan Özellikler:
- 🚀 **NVIDIA GPU NVENC Hızlandırma:** Google Colab GPU (T4/V100/A100/L4) otomatik algılanır; render işlemleri CPU'ya kıyasla **10x - 30x daha hızlı** gerçekleşir.
- ☁️ **Google Drive Entegrasyonu:** Görev veritabanı (`tasks_db.json`), biten videolar (`outputs/`), sesler ve ayarlar (`settings.json`) Drive'da kalıcı olarak saklanır.
- 🔄 **Kaldığı Yerden Devam Etme (Auto-Resume):** Colab kapandığında veya bağlantı koptuğunda, tekrar açtığınızda kaldığı yerden otomatik devam eder. Biten videolar asla kaybolmaz ve mükerrer render edilmez.
- ⚙️ **Otomatik Ayar & API Anahtarı Yönetimi:** Drive'da `settings.json` yoksa otomatik oluşturulur.
- 🌐 **Cloudflare / Ngrok Tüneli:** Tek tıkla şifresiz, güvenli web arayüzü bağlantısı.
- 📝 **Toplu Headless Üretim:** Web arayüzünü açık tutmadan Google Drive klasörüne attığınız onlarca ders metnini arka planda otomatik videolaştırır.

### 📁 1. Adım: Google Drive'ı Bağlama ve Ortam Kurulumu
*Google Drive hesabınızı bağlayarak tüm verilerin kalıcı olmasını sağlayın.*

In [ ]:
# @title 📁 1. Adım: Google Drive Bağlama
import os
from google.colab import drive

# 1. Google Drive'ı bağla
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/MoneyPrinterTurbo'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Google Drive başarıyla bağlandı! Çalışma klasörü: {DRIVE_DIR}')

### 📦 2. Adım: Depoyu Klonlama, Paketleri Yükleme ve GPU NVENC Kurulumu
*FFmpeg GPU (NVENC), Cloudflared ve Python bağımlılıklarını kurar.*

In [ ]:
# @title 📦 2. Adım: Kurulum & GPU NVENC Hızlandırıcı
import os, sys, shutil

# GPU Kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'CPU Modu Aktif'

# Depoyu klonla veya güncelle
if not os.path.exists('/content/MoneyPrinterTurbo-Lite'):
    !git clone https://github.com/TheOsmanYILDIRIM/MoneyPrinterTurbo-Lite.git /content/MoneyPrinterTurbo-Lite
else:
    %cd /content/MoneyPrinterTurbo-Lite
    !git pull

%cd /content/MoneyPrinterTurbo-Lite
sys.path.insert(0, '/content/MoneyPrinterTurbo-Lite')

# 1. Temel paketleri ve Cloudflared kur
!apt-get update -qq && apt-get install -y -qq ffmpeg xz-utils
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!pip install -q -r requirements-lite.txt pyngrok

# 2. Colab Tesla T4 GPU ile tam uyumlu FFmpeg 7.1 NVENC binary'sini kur
!if command -v nvidia-smi &> /dev/null; then \
    echo '⚡ Colab T4 GPU için NVENC 13.0 uyumlu FFmpeg 7.1 yapılandırılıyor...'; \
    wget -q -nc -O /tmp/ffmpeg-nvenc.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/autobuild-2026-08-16-13-00/ffmpeg-n7.1.5-16-g9a4bb2c579-linux64-gpl-7.1.tar.xz && \
    mkdir -p /tmp/ffmpeg_extracted && \
    tar -xf /tmp/ffmpeg-nvenc.tar.xz -C /tmp/ffmpeg_extracted && \
    cp -f /tmp/ffmpeg_extracted/*/bin/ffmpeg /usr/local/bin/ && \
    cp -f /tmp/ffmpeg_extracted/*/bin/ffprobe /usr/local/bin/ && \
    chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe && \
    rm -rf /tmp/ffmpeg_extracted /tmp/ffmpeg-nvenc.tar.xz && \
    echo '✅ NVENC FFmpeg başarıyla yüklendi!'; \
fi

# 3. Google Drive Entegrasyonunu ve GPU Encoder'ı Başlat
import drive_sync, lite_engine
drive_sync.init_drive_environment(drive_dir='/content/drive/MyDrive/MoneyPrinterTurbo')
# Encoder önbelleğini temizleyip yeniden tara
lite_engine._ENCODER_CONFIG = None
enc = lite_engine.get_video_encoder_config()
print(f"⚡ Aktif Video Encoder: {enc['name']} ({'NVIDIA GPU Hızlandırmalı 🚀' if enc.get('is_gpu') else 'CPU Modu'})")

### ⚙️ 3. Adım: Ayarlar ve API Anahtarlarını Yapılandırma (İsteğe Bağlı)
*API anahtarlarınızı doğrudan Google Drive'daki `settings.json` dosyasına kaydeder. Boş bıraktığınız alanlar mevcut ayarlarınızı korur.*

In [ ]:
# @title ⚙️ 3. Adım: API Anahtarları ve Video Ayarları Formu
import settings_manager

# @markdown **🔑 API Anahtarları:**
PEXELS_API_KEY = "" # @param {type:"string"}
PIXABAY_API_KEY = "" # @param {type:"string"}
GEMINI_API_KEY = "" # @param {type:"string"}
OPENAI_API_KEY = "" # @param {type:"string"}
GROQ_API_KEY = "" # @param {type:"string"}
AZURE_SPEECH_KEY = "" # @param {type:"string"}
AZURE_SPEECH_REGION = "eastus" # @param {type:"string"}
ELEVENLABS_API_KEY = "" # @param {type:"string"}
NGROK_AUTHTOKEN = "" # @param {type:"string"}

# @markdown **🎨 Varsayılan Video Tercihleri:**
DEFAULT_VOICE = "tr-TR-AhmetNeural" # @param ["tr-TR-AhmetNeural", "tr-TR-EmelNeural", "en-US-GuyNeural", "en-US-JennyNeural"]
DEFAULT_ASPECT = "9:16" # @param ["9:16", "16:9", "1:1"]
DEFAULT_BG_STYLE = "chalkboard" # @param ["chalkboard", "dark_slate", "warm_study", "pexels"]

updates = {}
if PEXELS_API_KEY: updates["pexels_api_keys"] = PEXELS_API_KEY
if PIXABAY_API_KEY: updates["pixabay_api_keys"] = PIXABAY_API_KEY
if GEMINI_API_KEY: updates["gemini_api_key"] = GEMINI_API_KEY
if OPENAI_API_KEY: updates["openai_api_key"] = OPENAI_API_KEY
if GROQ_API_KEY: updates["groq_api_key"] = GROQ_API_KEY
if AZURE_SPEECH_KEY: updates["azure_speech_key"] = AZURE_SPEECH_KEY
if AZURE_SPEECH_REGION: updates["azure_speech_region"] = AZURE_SPEECH_REGION
if ELEVENLABS_API_KEY: updates["elevenlabs_api_key"] = ELEVENLABS_API_KEY
if NGROK_AUTHTOKEN: updates["ngrok_authtoken"] = NGROK_AUTHTOKEN

if DEFAULT_VOICE: updates["prod_voice"] = DEFAULT_VOICE
if DEFAULT_ASPECT: updates["prod_aspect"] = DEFAULT_ASPECT
if DEFAULT_BG_STYLE: updates["prod_bg_style"] = DEFAULT_BG_STYLE

if updates:
    settings_manager.save_settings(updates)
    print("✅ Ayarlar Google Drive'a başarıyla kaydedildi!")

print("\n📋 Google Drive'daki Mevcut Ayarlar:")
for k, v in settings_manager.get_masked_settings().items():
    if v:
        print(f"  • {k}: {v}")

### 🌐 4. Adım: WebUI Studio Sunucusunu Başlatma (Önerilen)
*Cloudflare veya Ngrok tüneli üzerinden WebUI arayüzünü açar. Tarayıcınızdan tekli ve toplu video üretimlerini kolayca yönetebilirsiniz. Otomatik devam etme (auto-resume) ve GPU hızlandırma etkindir.*

In [ ]:
# @title 🌐 4. Adım: WebUI Başlatıcı
!fuser -k 8080/tcp >/dev/null 2>&1 || true
TUNNEL_TYPE = "cloudflare" # @param ["cloudflare", "ngrok", "none"]
AUTO_RESUME = True # @param {type:"boolean"}

resume_flag = "--auto-resume" if AUTO_RESUME else ""
!python lite_server.py --host 0.0.0.0 --port 8080 --tunnel {TUNNEL_TYPE} {resume_flag} --storage-dir "/content/drive/MyDrive/MoneyPrinterTurbo"

### 🚀 5. Adım: Toplu Headless Video Üretimi (WebUI Olmadan)
*Google Drive içindeki `batch_inputs/` klasörüne eklediğiniz tüm ders metinlerini sırayla render eder. Colab bağlantısı kopsa bile tekrar çalıştırdığınızda tam kaldığı yerden devam eder.*

In [ ]:
# @title 🚀 5. Adım: Toplu Headless Render & Auto-Resume
!python batch_processor.py

### 📊 6. Adım: Google Drive Görev & Biten Video Durum Raporu
*Google Drive'daki tüm tamamlanan videoları, boyutlarını ve işlem durumlarını listeler.*

In [ ]:
# @title 📊 6. Adım: Google Drive Durumunu Kontrol Et
import drive_sync
drive_sync.print_drive_status()